In [28]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

In [29]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [30]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [31]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    # renombrar a opción
    df.ejercicio = df.ejercicio.str.replace(". Opción", ", opción")

    # limpiar tema
    df.tema = df.tema.str.replace("Conf. Electrónica", "Configuración electrónica")
    df.tema = df.tema.str.lower().str.strip()

    return df

Arreglar ejercicios de Ponencia

In [32]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [33]:
df = df.apply(fix_ponencia, axis = 1)

In [34]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...,NaN


In [35]:
print(df[df.ejercicio.isna()].info_ejercicio)

2556    MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...
Name: info_ejercicio, dtype: str


In [36]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [37]:
df.sample(3)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2704,Mates II,Mates II,2021,Julio,Ejercicio 4,Integrales,MATEMÁTICAS II. 2021. JULIO. EJERCICIO 4,x 2 1\nConsidera la función f definida f(x) ...,source_pdf_files/Mates II/integrales/2021 - In...,enunciados_ejercicios/Mates II/integrales/png/...,NaN
4176,Química,Química,2017,Junio,Ejercicio 5. Opc Ión A,Equilibrio Químico,QUÍMICA. 2017. JUNIO. EJERCICIO 5. OPC IÓN A,"Para el equilibrio: H (g)CO (g) H O(g)CO(g),...",source_pdf_files/Química/equilibrio_químico/20...,enunciados_ejercicios/Química/equilibrio_quími...,NaN
2458,Mates II,Mates II,2015,Septiembre,Ejercicio 3. Opción B,Sistemas De Ecuaciones Lineales,MATEMÁTICAS II. 2015. SEPTIEMBRE. EJERCICIO 3....,2x y(1)z  1 \n\nConsidera el siguient...,source_pdf_files/Mates II/sistemas_de_ecuacion...,enunciados_ejercicios/Mates II/sistemas_de_ecu...,NaN


Limpiar los enunciados

In [38]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [39]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    # se usa str() para convertir año (int) en string
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [40]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [41]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [42]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
52,fisica_campo_electrico_y_magnetico_2021_julio_...,Física,Física,2021,Julio,Ejercicio B2,campo eléctrico y magnético,FISICA. 2021. JULIO. EJERCICIO B2,"a) Suponga dos conductores rectilíneos, muy la...",source_pdf_files/Física/campo_eléctrico_magnét...,enunciados_ejercicios/Física/campo_eléctrico_m...
3125,quimica_configuracion_electronica_2024_reserva...,Química,Química,2024,Reserva 4,Ejercicio B2,configuración electrónica,QUIMICA. 2024. RESERVA 4. EJERCICIO B2,El elemento He precede al Li en la tabla perió...,source_pdf_files/Química/configuración_electró...,enunciados_ejercicios/Química/configuración_el...
466,fisica_optica_geometrica_2021_junio_ejercicio_c2,Física,Física,2021,Junio,Ejercicio C2,óptica geométrica,FISICA. 2021. JUNIO. EJERCICIO C2,a) Con una lente queremos obtener una imagen v...,source_pdf_files/Física/óptica_geométrica/2021...,enunciados_ejercicios/Física/óptica_geométrica...
2286,mates_ii_geometria_2003_junio_ejercicio_3_opci...,Mates II,Mates II,2003,Junio,"Ejercicio 3, opción A",geometría,MATEMÁTICAS II. 2003. JUNIO. EJERCICIO 3. OPCI...,"  \nConsidera los vectores u (1,1,1), v (...",source_pdf_files/Mates II/geometría/2003 - Geo...,enunciados_ejercicios/Mates II/geometría/png/p...
1548,mates_ccss_programacion_lineal_2020_reserva_4_...,Mates CCSS,Mates CCSS,2020,Reserva 4,Ejercicio A2,programación lineal,SOCIALES II. 2020 RESERVA 4. EJERCICIO A2,Se considera la región del plano definida por ...,source_pdf_files/Mates CCSS/programación_linea...,enunciados_ejercicios/Mates CCSS/programación_...


In [43]:
df.isna().sum()

id                    0
asignatura            0
asignatura_carpeta    0
año                   0
convocatoria          0
ejercicio             0
tema                  0
info_ejercicio        0
enunciado             4
archivo               0
imagen                0
dtype: int64

In [44]:
if all(df.asignatura == df.asignatura_carpeta):
    df.drop("asignatura_carpeta", axis = 1, inplace=True)

In [22]:
df.shape

(4774, 10)

In [23]:
df[df.enunciado.isna()].shape

(4, 10)

In [45]:
df.asignatura.unique()

<ArrowStringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [46]:
df.loc[:, ["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,campo eléctrico y magnético
121,Física,campo gravitatorio
244,Física,física cuántica y nuclear
362,Física,ondas
449,Física,óptica geométrica
486,Mates CCSS,contraste de hipótesis
545,Mates CCSS,distribución binomial
550,Mates CCSS,funciones ccss
806,Mates CCSS,inferencia estadística
1078,Mates CCSS,matrices y determinantes


In [48]:
df.convocatoria.unique()

<ArrowStringArray>
[             'Julio',              'Junio',          'Reserva 1',
          'Reserva 2',          'Reserva 3',          'Reserva 4',
         'Septiembre', 'Reserva1 Ejercicio',           'Ponencia',
             'Modelo']
Length: 10, dtype: str

In [49]:
df.loc[df.convocatoria == "Reserva1 Ejercicio",
       ["convocatoria", "ejercicio"]] = ("Reserva 1", "Ejercicio 1")

In [50]:
df.to_csv(root/"csv_enunciados/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [51]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [52]:
df.sample(3)

,id,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
2021,mates_ii_geometria_2026_junio_ejercicio_2,Mates II,2026,Junio,Ejercicio 2,geometría,MATEMÁTICAS II. 2026. JUNIO. EJERCICIO 2,"Considera los puntos A ( 1 , 2 , 0 ) , B ( 2 ,...",source_pdf_files/Mates II/geometría/2026 - Geo...,enunciados_ejercicios/Mates II/geometría/png/p...
2056,mates_ii_geometria_2023_reserva_4_ejercicio_8,Mates II,2023,Reserva 4,Ejercicio 8,geometría,MATEMÁTICAS II. 2023. RESERVA 4. EJERCICIO 8,x yz  1\nConsidera la recta r \n 3x2z ...,source_pdf_files/Mates II/geometría/2023 - Geo...,enunciados_ejercicios/Mates II/geometría/png/p...
3721,quimica_formulacion_2019_reserva_2_ejercicio_1...,Química,2019,Reserva 2,"Ejercicio 1, opción A",formulación,QUÍMICA. 2019. RESERVA 2. EJERCICIO 1. OPCIÓN A,Formule o nombre los siguientes compuestos: a)...,source_pdf_files/Química/formulación/2019 - Fo...,enunciados_ejercicios/Química/formulación/png/...
